In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from pathlib import Path
import sys
import pyagrum as gum
import pyagrum.lib.notebook as gnb
from IPython.display import display, HTML
import warnings
from tqdm import tqdm

sys.path.insert(0, str(Path().resolve().parents[1]))
from src.config import *  # noqa
from src.utils import *  # noqa
from src.mosaic import *  # noqa

In [ ]:
def plot_cset(bn_min_max: tuple, var: str, parents_idx: int, ax, kwargs: dict = {}):

    # Extract the CPT
    bn_min, bn_max = bn_min_max
    cpt_min, cpt_max = get_tabular_cpt(bn_min.cpt(var)), get_tabular_cpt(
        bn_max.cpt(var)
    )

    # Extract the row corresp. to `parents`
    row = [cpt_min[parents_idx, :], cpt_max[parents_idx, :]]

    # Plot
    ax.plot([row[0][0], row[1][0]], [row[1][1], row[0][1]], **kwargs)


def plot_point(bn: gum.BayesNet, var: str, parents_idx: int, ax, kwargs: dict = {}):

    # Extract the info
    cpt = get_tabular_cpt(bn.cpt(var))
    row = cpt[parents_idx, :]

    # Plot
    ax.scatter(row[0], row[1], **kwargs)

### Read results

In [ ]:
# Read config file
config = load_config("conf.yaml")

# Choose results path
res_path = Path("results_0.1")

# Read baseline BN
bn_base = gum.loadBN(config["bn_base_path"])
confs = get_confs(bn_base)

# Read sample sizes and repetitions
sizes_dict = config["s_sizes"]
sizes = [
    int(x) for x in np.arange(sizes_dict["min"], sizes_dict["max"], sizes_dict["step"])
]
n_reps = config["n_repetitions"]

In [ ]:
# Init results
res_mle = pd.DataFrame({"size": sizes})
res_idm = pd.DataFrame({"size": sizes})
res_upd = pd.DataFrame({"size": sizes})

# For every sample size ...
for idx, n in enumerate(tqdm(sizes)):

    ## Read folder
    base_path = res_path / f"ss{n}"

    # BN
    bn = [gum.loadBN(f"{base_path}/{rep}-bn.bif") for rep in range(n_reps)]  #

    # CN
    idm_bn_min = [
        gum.loadBN(f"{base_path}/{rep}-idm_bn_min.bif") for rep in range(n_reps)
    ]
    idm_bn_max = [
        gum.loadBN(f"{base_path}/{rep}-idm_bn_max.bif") for rep in range(n_reps)
    ]
    idm = np.array(list(zip(idm_bn_min, idm_bn_max)), dtype=object)

    # Mosaic
    mos_bn_min = [
        gum.loadBN(f"{base_path}/{rep}-mos_bn_min.bif") for rep in range(n_reps)
    ]
    mos_bn_max = [
        gum.loadBN(f"{base_path}/{rep}-mos_bn_max.bif") for rep in range(n_reps)
    ]

    # For every configuration ...
    for var, parents in confs:

        kl_mle = np.array([get_kl(x, bn_base, var, parents) for x in bn])
        kl_idm = np.array(
            [
                get_kl_cset((x, y), bn_base, var, parents)
                for x, y in zip(idm_bn_min, idm_bn_max)
            ]
        )
        kl_upd = np.array(
            [
                get_kl_cset((x, y), bn_base, var, parents)
                for x, y in zip(mos_bn_min, mos_bn_max)
            ]
        )

        # Save results
        # kl = {"kl_mle":kl_mle, "kl_idm":kl_idm, "kl_upd":kl_upd}
        colname = f"{var}:{parents}"
        for df in [res_mle, res_idm, res_upd]:
            if colname not in df.columns:
                df[colname] = None
        res_mle.at[idx, colname] = kl_mle
        res_idm.at[idx, colname] = kl_idm
        res_upd.at[idx, colname] = kl_upd

In [ ]:
# Save results
res_mle.to_parquet(res_path / f"res_mle.parquet")
res_idm.to_parquet(res_path / f"res_idm.parquet")
res_upd.to_parquet(res_path / f"res_upd.parquet")

In [ ]:
# Read results
res_mle = pd.read_parquet(res_path / f"res_mle.parquet")
res_idm = pd.read_parquet(res_path / f"res_idm.parquet")
res_upd = pd.read_parquet(res_path / f"res_upd.parquet")

In [ ]:
# Choose results to plot
df1, df2 = res_mle, res_upd

diff = df1.iloc[:, 1:] - df2.iloc[:, 1:]
res = pd.concat([df1.iloc[:, :1], diff], axis=1)

### Plot KL

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(15, 6), constrained_layout=True)
axs_flat = axs.flatten()

ax_idx = 0


def col(vec):
    return ["green" if v > 0 else "red" for v in vec]


# For each pair of (var, parents) ...
for var, parents in confs:

    # Retrieve the results
    colname = var + ":" + str(parents)
    vec = res[colname]

    # Plot
    ax = axs_flat[ax_idx]
    ax_idx += 1

    ## Scatter plot
    # color = ["green" if x>=0 else "red" for x in vec]
    # ax.scatter(sizes, vec, s=20, color=color, alpha=.7)

    ## Box plot
    data = res[colname]
    labels = res["size"].tolist()
    ax.boxplot(
        data,
        tick_labels=labels,
        patch_artist=True,
        boxprops=dict(facecolor="steelblue", alpha=0.4),
        medianprops=dict(color="red", linewidth=2),
    )
    ax.axhline(y=0, color="black", linestyle="--", linewidth=1)
    ax.set_title(colname)

In [ ]:
gnb.showCPTs(bn_base)